# GeoDiff-GAN DGX 25MCSA19: Captioned vs No-Caption Variants

This is a copy of the multispectral variant workflow for the assigned DGX folder only:
`/workspace/temp/25mcsa19/working/temp/25mcsa19`.

It does **not** overwrite the already-trained no-caption models. It trains four new
caption-conditioned variants in separate folders:

- `small_improved_caption`
- `medium_caption`
- `small_improved_ms_caption`
- `medium_ms_caption`

Then it compares all eight variants:

- four existing no-caption variants
- four new caption-conditioned variants

Metrics: PSNR, SSIM, Edge F1, and LPIPS.

## 0. Create the local Python 3.11 environment

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

ASSIGNED_ROOT = Path("/workspace/temp/25mcsa19/working/temp/25mcsa19").resolve()
ASSIGNED_ROOT.mkdir(parents=True, exist_ok=True)
VENV = ASSIGNED_ROOT / ".venvs" / "geodiff-py311"
KERNEL_DIR = ASSIGNED_ROOT / "jupyter_kernels" / "geodiff-py311-25mcsa19"

def run(command, check=True, cwd=None):
    command = [str(value) for value in command]
    print("+", " ".join(command), flush=True)
    return subprocess.run(command, check=check, cwd=cwd)

uv = shutil.which("uv")
if uv is None:
    BOOTSTRAP = ASSIGNED_ROOT / ".bootstrap"
    run([sys.executable, "-m", "pip", "install", "--prefix", BOOTSTRAP, "uv"])
    uv = str(BOOTSTRAP / "bin" / "uv")
if not Path(uv).exists() and shutil.which(uv) is None:
    raise RuntimeError(f"uv was not found at {uv}. Ask the administrator to provide uv or Python 3.11.")

run([uv, "python", "install", "3.11"])
run([uv, "venv", "--python", "3.11", "--seed", VENV])
PYTHON311 = VENV / "bin" / "python"
run([PYTHON311, "-m", "ensurepip", "--upgrade"], check=False)
run([PYTHON311, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel", "ipykernel"])

KERNEL_DIR.mkdir(parents=True, exist_ok=True)
(KERNEL_DIR / "kernel.json").write_text(json.dumps({
    "argv": [str(PYTHON311), "-m", "ipykernel_launcher", "-f", "{connection_file}"],
    "display_name": "GeoDiff-GAN 25MCSA19 Python 3.11",
    "language": "python",
    "env": {
        "PYTHONUNBUFFERED": "1",
        "XDG_CACHE_HOME": str(ASSIGNED_ROOT / ".cache"),
        "HF_HOME": str(ASSIGNED_ROOT / ".cache" / "huggingface"),
        "TORCH_HOME": str(ASSIGNED_ROOT / ".cache" / "torch"),
        "KAGGLE_CONFIG_DIR": str(ASSIGNED_ROOT / "secrets" / "kaggle"),
        "KAGGLEHUB_CACHE": str(ASSIGNED_ROOT / "downloads" / "kagglehub_cache")
    }
}, indent=2), encoding="utf-8")

print("Created Python:", PYTHON311)
print("Created local kernelspec:", KERNEL_DIR)
print("No files outside the assigned root were modified by default.")
print()
print("If this kernel is not visible in Jupyter, run this optional command manually:")
print(f"{PYTHON311} -m ipykernel install --user --name geodiff-py311-25mcsa19 --display-name 'GeoDiff-GAN 25MCSA19 Python 3.11'")
print("Then switch Kernel ->", "GeoDiff-GAN 25MCSA19 Python 3.11", "and rerun from the runtime cell.")

## 1. Runtime root guard

Switch to the Python 3.11 kernel created above, then run from here.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

ASSIGNED_ROOT = Path("/workspace/temp/25mcsa19/working/temp/25mcsa19").resolve()
if not str(ASSIGNED_ROOT).startswith("/workspace/temp/25mcsa19/working/temp/25mcsa19"):
    raise RuntimeError(f"Unsafe root: {ASSIGNED_ROOT}")
ASSIGNED_ROOT.mkdir(parents=True, exist_ok=True)

THESIS_ROOT = ASSIGNED_ROOT
REPOSITORY_DIR = THESIS_ROOT / "geodiff-gan"
DOWNLOAD_ROOT = THESIS_ROOT / "downloads"
DATASET_ROOT = THESIS_ROOT / "datasets" / "sentinel2-bharat"
WORK_ROOT = THESIS_ROOT / "geodiff-output"
SOURCE_ROOT = THESIS_ROOT / "sota_sources"
BACKUP_ROOT = THESIS_ROOT / "backups"
REPOSITORY_URL = "https://github.com/shashankjs2002/SI-SR-1.git"
KAGGLE_DATASET = "twilight2002/sentinel2-bharat"

for path in (DOWNLOAD_ROOT, DATASET_ROOT, WORK_ROOT, SOURCE_ROOT, BACKUP_ROOT):
    path.mkdir(parents=True, exist_ok=True)

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["XDG_CACHE_HOME"] = str(THESIS_ROOT / ".cache")
os.environ["HF_HOME"] = str(THESIS_ROOT / ".cache" / "huggingface")
os.environ["TRANSFORMERS_CACHE"] = str(THESIS_ROOT / ".cache" / "huggingface")
os.environ["TORCH_HOME"] = str(THESIS_ROOT / ".cache" / "torch")
os.environ["KAGGLE_CONFIG_DIR"] = str(THESIS_ROOT / "secrets" / "kaggle")
os.environ["KAGGLEHUB_CACHE"] = str(DOWNLOAD_ROOT / "kagglehub_cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def assert_inside(path):
    path = Path(path).resolve()
    if not str(path).startswith(str(THESIS_ROOT)):
        raise RuntimeError(f"Refusing to touch path outside assigned root: {path}")
    return path

def run(command, cwd=None, env=None, check=True):
    command = [str(value) for value in command]
    environment = os.environ.copy()
    if env:
        environment.update(env)
    print("+", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, env=environment, check=check)

PYTHON = Path(sys.executable)
PIP = [PYTHON, "-m", "pip"]
print("Python:", sys.version)
print("Executable:", PYTHON)
print("Assigned root:", THESIS_ROOT)
if sys.version_info < (3, 10):
    raise RuntimeError("Switch to the GeoDiff-GAN 25MCSA19 Python 3.11 kernel before continuing.")

## 2. Clone/update repository and install dependencies

In [ ]:
import os, sys
from pathlib import Path

if REPOSITORY_DIR.exists():
    if (REPOSITORY_DIR / ".git").exists():
        print("Updating existing Git clone:", REPOSITORY_DIR)
        run(["git", "pull", "--ff-only"], cwd=REPOSITORY_DIR, check=False)
    elif (REPOSITORY_DIR / "pyproject.toml").exists() and (REPOSITORY_DIR / "src").exists():
        print("Using existing uploaded source tree:", REPOSITORY_DIR)
    else:
        raise RuntimeError(f"{REPOSITORY_DIR} exists but is not a GeoDiff-GAN source tree.")
else:
    run(["git", "clone", "--depth", "1", REPOSITORY_URL, REPOSITORY_DIR])

def patch_geodiff_source_compatibility():
    # Patch older cloned source trees inside the assigned DGX folder only.
    system_path = REPOSITORY_DIR / "src" / "geodiff_gan" / "models" / "system.py"
    parameters_path = REPOSITORY_DIR / "src" / "geodiff_gan" / "parameters.py"
    for path in (system_path, parameters_path):
        assert_inside(path)
        if not path.exists():
            raise FileNotFoundError(path)

    system_text = system_path.read_text(encoding="utf-8")
    changed = False
    if "self.output_channels = output_channels" not in system_text:
        if "self.input_channels = input_channels\n" in system_text:
            system_text = system_text.replace(
                "        self.input_channels = input_channels\n",
                "        self.input_channels = input_channels\n"
                "        self.output_channels = output_channels\n",
                1,
            )
        else:
            system_text = system_text.replace(
                "        self.scale = scale\n",
                "        self.scale = scale\n"
                "        self.input_channels = input_channels\n"
                "        self.output_channels = output_channels\n",
                1,
            )
        changed = True
    if changed:
        system_path.write_text(system_text, encoding="utf-8")
        print("Patched GeoDiffGAN input/output channel attributes:", system_path)

    parameters_text = parameters_path.read_text(encoding="utf-8")
    if 'model_channels = config.get("model", config)' not in parameters_text:
        needle = '    model = GeoDiffGAN.from_config(config)\n'
        replacement = (
            '    model = GeoDiffGAN.from_config(config)\n'
            '    model_channels = config.get("model", config)\n'
            '    output_channels = int(\n'
            '        getattr(model, "output_channels", model_channels.get("output_channels", 3))\n'
            '    )\n'
        )
        parameters_text = parameters_text.replace(needle, replacement, 1)
        parameters_text = parameters_text.replace(
            '        output_channels=model.output_channels,\n',
            '        output_channels=output_channels,\n',
        )
        parameters_text = parameters_text.replace(
            '        condition_channels=model.output_channels,\n',
            '        condition_channels=output_channels,\n',
        )
        parameters_path.write_text(parameters_text, encoding="utf-8")
        print("Patched parameter-report output channel fallback:", parameters_path)

patch_geodiff_source_compatibility()

run([*PIP, "install", "--upgrade", "pip", "setuptools", "wheel"])
run([*PIP, "install", "numpy>=1.26", "Pillow>=10", "PyYAML>=6", "tqdm>=4.66", "rasterio>=1.3", "pandas>=2", "matplotlib>=3.8", "kaggle", "kagglehub", "ipykernel", "transformers>=4.57", "sentencepiece>=0.2", "protobuf>=4.25", "accelerate>=1.0", "safetensors>=0.4"])
run([*PIP, "install", "-e", ".", "--no-deps"], cwd=REPOSITORY_DIR)
sys.path.insert(0, str(REPOSITORY_DIR / "src"))
os.chdir(REPOSITORY_DIR)
run(["git", "log", "-1", "--oneline"], cwd=REPOSITORY_DIR, check=False)

## 3. Verify the allocated A100 GPU

In [ ]:
import torch

print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available(), "visible GPUs:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable. Ask the administrator to check the DGX Jupyter container.")
if torch.cuda.device_count() != 1:
    raise RuntimeError("Exactly one GPU must be visible. Coordinate with other users and set CUDA_VISIBLE_DEVICES before starting.")
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, "VRAM GiB:", round(props.total_memory / 1024**3, 2))
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

## 4. Kaggle credentials and dataset download inside the assigned folder

In [ ]:
from getpass import getpass
import json, os, zipfile

KAGGLE_DIR = THESIS_ROOT / "secrets" / "kaggle"
KAGGLE_JSON = KAGGLE_DIR / "kaggle.json"
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_DIR)

if not KAGGLE_JSON.exists():
    username = input("Kaggle username: ").strip()
    key = getpass("Kaggle API key: ").strip()
    if not username or not key:
        raise RuntimeError("Kaggle credentials not provided")
    KAGGLE_JSON.write_text(json.dumps({"username": username, "key": key}), encoding="utf-8")
    KAGGLE_JSON.chmod(0o600)

DOWNLOAD_DATASET = True
DOWNLOAD_METHOD = "kaggle_cli"  # kaggle_cli | kagglehub

if DOWNLOAD_DATASET and not list(DATASET_ROOT.rglob("*.SAFE")):
    if DOWNLOAD_METHOD == "kaggle_cli":
        run([PYTHON, "-m", "kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", DOWNLOAD_ROOT])
        archives = sorted(DOWNLOAD_ROOT.glob("*.zip"))
        if not archives:
            raise RuntimeError("Kaggle CLI did not produce a zip archive")
        archive = archives[-1]
        with zipfile.ZipFile(archive) as handle:
            bad = handle.testzip()
            if bad is not None:
                raise RuntimeError(f"Corrupt zip member: {bad}")
            handle.extractall(DATASET_ROOT)
        print("Extracted:", archive, "->", DATASET_ROOT)
    elif DOWNLOAD_METHOD == "kagglehub":
        import kagglehub
        path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        print("kagglehub path:", path)
        DATASET_ROOT = path
    else:
        raise ValueError(DOWNLOAD_METHOD)
else:
    print("Reusing dataset:", DATASET_ROOT)

safe_count = len(list(DATASET_ROOT.rglob("*.SAFE")))
print("SAFE directories:", safe_count)
if safe_count == 0:
    raise RuntimeError(f"No SAFE products found below {DATASET_ROOT}")

## 5. Incremental multispectral Sentinel-2 preprocessing

In [ ]:
from collections import Counter
import json
from geodiff_gan.data import sentinel as sentinel_module

PATCH_SIZE = 512
PATCH_STRIDE = 384
MINIMUM_VALID_FRACTION = 0.95
MAX_PRODUCTS = None       # set small number for smoke test
REBUILD_PATCHES = False
PATCH_ROOT = WORK_ROOT / "patches_ms"
RAW_MANIFEST = WORK_ROOT / "manifest_ms_raw.jsonl"
PREPARATION_STATE = WORK_ROOT / "preparation-ms-state.json"

safe_candidates = sorted(DATASET_ROOT.rglob("*.SAFE"))
safe_products = sentinel_module.discover_safe_products(DATASET_ROOT)
print(f"Canonical={len(safe_products)}, scanned={len(safe_candidates)}, wrappers ignored={len(safe_candidates)-len(safe_products)}")
for product in safe_products[:20]:
    print(" -", sentinel_module.source_product_name(product), "->", product)
if not safe_products:
    raise RuntimeError(f"No canonical SAFE product below {DATASET_ROOT}")

command = [
    PYTHON, "-m", "geodiff_gan.cli.prepare_sentinel",
    "--input", DATASET_ROOT,
    "--output", PATCH_ROOT,
    "--manifest", RAW_MANIFEST,
    "--state", PREPARATION_STATE,
    "--patch-size", PATCH_SIZE,
    "--stride", PATCH_STRIDE,
    "--minimum-valid-fraction", MINIMUM_VALID_FRACTION,
    "--multispectral-preset", "rgb-nir-swir",
    "--unmatched-split", "train",
]
if MAX_PRODUCTS is not None:
    command.extend(["--max-products", MAX_PRODUCTS])
if REBUILD_PATCHES:
    command.append("--rebuild")
for historical in (WORK_ROOT / "manifest_ms.before-edge-filter.jsonl", WORK_ROOT / "rejected-ms-edge-patches.jsonl"):
    if historical.exists():
        command.extend(["--completed-manifest", historical])
run(command, cwd=REPOSITORY_DIR)

records = [json.loads(line) for line in RAW_MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()]
print("Patches:", len(records), "products:", len({r["source_product"] for r in records}), "tiles:", Counter(r["tile_id"] for r in records))

## 6. Quarantine corrupt/black edge patches

In [ ]:
import json, shutil
import numpy as np
from collections import Counter

MANIFEST = RAW_MANIFEST
BACKUP_MANIFEST = WORK_ROOT / "manifest_ms.before-edge-filter.jsonl"
REJECTED_MANIFEST = WORK_ROOT / "rejected-ms-edge-patches.jsonl"
QUARANTINE_ROOT = WORK_ROOT / "quarantine_ms_edge_patches"
BLACK_THRESHOLD = 1e-6
EDGE_WIDTH = 32
CORNER_SIZE = 64
MAX_EDGE_BLACK_FRACTION = 0.002
MAX_CORNER_BLACK_FRACTION = 0.002
MAX_OVERALL_BLACK_FRACTION = 0.005

active_records = [json.loads(line) for line in MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()]
BACKUP_MANIFEST.write_text("".join(json.dumps(record) + "\n" for record in active_records), encoding="utf-8")

def inspect_patch(path):
    path = Path(path)
    if not path.exists():
        return True, {"reason": "missing_patch"}
    try:
        with np.load(path) as data:
            hr = np.asarray(data["hr"], dtype=np.float32)
            if "ms_hr" not in data:
                return True, {"reason": "missing_ms_hr"}
            ms_hr = np.asarray(data["ms_hr"], dtype=np.float32)
    except Exception as error:
        return True, {"reason": f"unreadable:{type(error).__name__}"}
    if hr.shape[0] != 3 or ms_hr.shape[0] != 6:
        return True, {"reason": f"unexpected_shape:hr={hr.shape},ms={ms_hr.shape}"}
    if not np.isfinite(hr).all() or not np.isfinite(ms_hr).all():
        return True, {"reason": "nonfinite"}
    black = np.all(hr <= BLACK_THRESHOLD, axis=0)
    h, w = black.shape
    edge = min(EDGE_WIDTH, h // 2, w // 2)
    corner = min(CORNER_SIZE, h // 2, w // 2)
    edge_pixels = np.concatenate([black[:edge].ravel(), black[-edge:].ravel(), black[edge:-edge, :edge].ravel(), black[edge:-edge, -edge:].ravel()])
    corner_pixels = np.concatenate([black[:corner, :corner].ravel(), black[:corner, -corner:].ravel(), black[-corner:, :corner].ravel(), black[-corner:, -corner:].ravel()])
    stats = {
        "overall_black_fraction": float(black.mean()),
        "edge_black_fraction": float(edge_pixels.mean()),
        "corner_black_fraction": float(corner_pixels.mean()),
    }
    reasons = []
    if stats["overall_black_fraction"] > MAX_OVERALL_BLACK_FRACTION:
        reasons.append("overall_black")
    if stats["edge_black_fraction"] > MAX_EDGE_BLACK_FRACTION:
        reasons.append("edge_black")
    if stats["corner_black_fraction"] > MAX_CORNER_BLACK_FRACTION:
        reasons.append("corner_black")
    stats["reason"] = ",".join(reasons) if reasons else "accepted"
    return bool(reasons), stats

kept, rejected = [], []
for index, record in enumerate(active_records, start=1):
    reject, stats = inspect_patch(record["patch"])
    if reject:
        rejected_record = dict(record)
        rejected_record["original_patch"] = record["patch"]
        rejected_record["filter"] = stats
        source = Path(record["patch"])
        if source.exists():
            destination = QUARANTINE_ROOT / record["tile_id"] / source.name
            destination.parent.mkdir(parents=True, exist_ok=True)
            if source.resolve() != destination.resolve():
                shutil.move(str(source), str(destination))
            rejected_record["patch"] = str(destination)
        rejected.append(rejected_record)
    else:
        kept.append(record)
    if index % 200 == 0 or index == len(active_records):
        print(f"Scanned {index}/{len(active_records)}")

MANIFEST.write_text("".join(json.dumps(record) + "\n" for record in kept), encoding="utf-8")
REJECTED_MANIFEST.write_text("".join(json.dumps(record) + "\n" for record in rejected), encoding="utf-8")
ACCEPTED_MANIFEST = MANIFEST
records = kept
print("Accepted:", len(kept), "Rejected:", len(rejected), "splits:", Counter(r["split"] for r in kept))
if not kept:
    raise RuntimeError("All patches were rejected.")

## 7. Deterministic 80/10/10 spatial-block split within every tile

In [ ]:
from collections import Counter, defaultdict
import hashlib, json, random

SPLIT_SEED = 20260708
SPLIT_MODE = "spatial_blocks"  # spatial_blocks | patch_random
SPATIAL_BLOCK_PIXELS = 2048
FINAL_MANIFEST = WORK_ROOT / "manifest_ms_80_10_10.jsonl"

def seed_for(tile):
    return int.from_bytes(hashlib.sha256(f"{SPLIT_SEED}:{tile}".encode()).digest()[:8], "little")

def assign(group):
    tile = group[0]["tile_id"]
    if SPLIT_MODE == "patch_random":
        random.Random(seed_for(tile)).shuffle(group)
        n = len(group)
        n_test = max(1, round(0.1 * n))
        n_val = max(1, round(0.1 * n))
        for i, record in enumerate(group):
            record["split"] = "test" if i < n_test else "val" if i < n_test + n_val else "train"
        return group
    blocks = defaultdict(list)
    for record in group:
        blocks[(record["row"] // SPATIAL_BLOCK_PIXELS, record["col"] // SPATIAL_BLOCK_PIXELS)].append(record)
    keys = list(blocks)
    random.Random(seed_for(tile)).shuffle(keys)
    targets = {"test": 0.1 * len(group), "val": 0.1 * len(group)}
    chosen = {}
    for split in ("test", "val"):
        count = 0
        while keys and (count < targets[split] or count == 0):
            key = keys.pop()
            chosen[key] = split
            count += len(blocks[key])
    for key in keys:
        chosen[key] = "train"
    output = []
    for key, values in blocks.items():
        for record in values:
            record["split"] = chosen[key]
            output.append(record)
    return output

accepted = [json.loads(line) for line in ACCEPTED_MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()]
by_tile = defaultdict(list)
for record in accepted:
    by_tile[record["tile_id"]].append(dict(record))
records = []
for tile, group in sorted(by_tile.items()):
    records.extend(assign(group))
FINAL_MANIFEST.write_text("".join(json.dumps(record) + "\n" for record in records), encoding="utf-8")
MANIFEST = FINAL_MANIFEST
counts = Counter(record["split"] for record in records)
print("Manifest:", FINAL_MANIFEST, "overall:", counts)
for tile in sorted(by_tile):
    values = {split: sum(r["tile_id"] == tile and r["split"] == split for r in records) for split in ("train", "val", "test")}
    total = sum(values.values())
    print(tile, values, {key: round(value / total, 3) for key, value in values.items()})
if any(counts[split] == 0 for split in ("train", "val", "test")):
    raise RuntimeError(f"Missing split: {counts}")

## 8. Visualize multispectral preparation samples

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.nn import functional as F
from geodiff_gan.models.degradation import random_degradation

DEGRADATION_SEVERITY = "mild"
DEGRADATION_SEED = 42
VISUALIZE_ACCEPTED = 5

accepted_records = [json.loads(line) for line in MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()]
selected = random.Random(42).sample(accepted_records, k=min(VISUALIZE_ACCEPTED, len(accepted_records)))

def chw_to_rgb(tensor):
    return tensor.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()

figure, axes = plt.subplots(len(selected), 5, figsize=(20, 4 * len(selected)), squeeze=False)
for row, record in enumerate(selected):
    with np.load(record["patch"]) as data:
        hr = torch.from_numpy(data["hr"]).float()
        ms_hr = torch.from_numpy(data["ms_hr"]).float()
    generator = torch.Generator().manual_seed(DEGRADATION_SEED + row)
    observed_lr, parameters, clean_lr = random_degradation(hr.unsqueeze(0), scale=4, generator=generator, return_clean=True, severity=DEGRADATION_SEVERITY)
    ms_lr, _, _ = random_degradation(ms_hr.unsqueeze(0), scale=4, generator=torch.Generator().manual_seed(DEGRADATION_SEED + row), return_clean=True, severity=DEGRADATION_SEVERITY)
    bicubic = F.interpolate(observed_lr, size=hr.shape[-2:], mode="bicubic", align_corners=False).clamp(0, 1)[0]
    nir_lr = ms_lr[0, 3:4].repeat(3, 1, 1)
    panels = [
        (hr, f"HR RGB\n{record['tile_id']} {record['split']}"),
        (clean_lr[0], "Clean RGB LR"),
        (observed_lr[0], "Observed RGB LR"),
        (nir_lr, "Observed NIR LR"),
        (bicubic, "Bicubic RGB x4"),
    ]
    for column, (image, title) in enumerate(panels):
        axes[row, column].imshow(chw_to_rgb(image))
        axes[row, column].set_title(title, fontsize=10)
        axes[row, column].axis("off")
plt.tight_layout()
plt.show()

## 9. Build Caption-Conditioned Configs

This cell requires the grounded caption file. Base and VAE are reused from the existing
no-caption runs by default because those stages do not use text conditioning. The captioned
training starts from the matching no-caption VAE checkpoint and trains `diffusion` and `joint`.

In [ ]:
import copy, yaml, json
from pathlib import Path
from geodiff_gan.config import load_config
from geodiff_gan.parameters import build_parameter_report

NO_CAPTION_VARIANTS = [
    "small_improved",
    "medium",
    "small_improved_ms",
    "medium_ms",
]

CAPTION_VARIANTS_TO_RUN = [
    "small_improved_caption",
    "medium_caption",
    "small_improved_ms_caption",
    "medium_ms_caption",
]

CAPTION_TO_BASE_VARIANT = {
    "small_improved_caption": "small_improved",
    "medium_caption": "medium",
    "small_improved_ms_caption": "small_improved_ms",
    "medium_ms_caption": "medium_ms",
}

ALL_VARIANTS_FOR_COMPARISON = NO_CAPTION_VARIANTS + CAPTION_VARIANTS_TO_RUN

# Reuse existing no-caption base/VAE checkpoints. This avoids retraining stages that do not use captions.
REUSE_NO_CAPTION_BASE_VAE = True
STAGES_TO_RUN_CAPTION = ["diffusion", "joint"] if REUSE_NO_CAPTION_BASE_VAE else ["base", "vae", "diffusion", "joint"]

CAPTION_JSONL = WORK_ROOT / "captions_grounded" / "captions_grounded_qwen3vl.jsonl"
if not CAPTION_JSONL.exists():
    raise FileNotFoundError(
        f"Caption file not found: {CAPTION_JSONL}. Run the grounded captioning notebook first."
    )

# Quick schema check. Old caption rows still work, but positional is expected for the new run.
caption_preview = []
with CAPTION_JSONL.open("r", encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            caption_preview.append(json.loads(line))
        if len(caption_preview) >= 20:
            break
if not caption_preview:
    raise RuntimeError(f"Caption file is empty: {CAPTION_JSONL}")
missing_positional = sum(
    "positional" not in row.get("captions", {}) for row in caption_preview
)
print("Caption file:", CAPTION_JSONL)
print("Preview rows:", len(caption_preview), "missing positional in preview:", missing_positional)
if missing_positional:
    print("WARNING: some old caption rows do not have captions.positional. They will still train using available fields.")

CONFIG_FILES_BY_BASE = {
    "small_improved": REPOSITORY_DIR / "configs" / "small_12tile_improved.yaml",
    "medium": REPOSITORY_DIR / "configs" / "medium.yaml",
    "small_improved_ms": REPOSITORY_DIR / "configs" / "small_12tile_improved_multispectral.yaml",
    "medium_ms": REPOSITORY_DIR / "configs" / "medium_multispectral.yaml",
}
DEFAULTS = REPOSITORY_DIR / "configs" / "default.yaml"
CONFIG_ROOT = WORK_ROOT / "configs"
RUN_ROOT = WORK_ROOT / "runs"

EPOCHS_BY_CAPTION_VARIANT = {
    "small_improved_caption": {"base": 8, "vae": 8, "diffusion": 20, "joint": 8},
    "medium_caption": {"base": 8, "vae": 8, "diffusion": 20, "joint": 8},
    "small_improved_ms_caption": {"base": 8, "vae": 8, "diffusion": 20, "joint": 8},
    "medium_ms_caption": {"base": 8, "vae": 8, "diffusion": 20, "joint": 8},
}

BATCH_BY_CAPTION_VARIANT = {
    "small_improved_caption": {"batch": 4, "accumulation": 4},
    "medium_caption": {"batch": 2, "accumulation": 8},
    "small_improved_ms_caption": {"batch": 4, "accumulation": 4},
    "medium_ms_caption": {"batch": 2, "accumulation": 8},
}

VALIDATION_LIMIT = 128
AUTO_RESUME_TRAINING = True


def build_caption_config(caption_variant):
    base_variant = CAPTION_TO_BASE_VARIANT[caption_variant]
    config = load_config(CONFIG_FILES_BY_BASE[base_variant], DEFAULTS)
    is_ms = base_variant.endswith("_ms")

    config["data"].update({
        "manifest": str(FINAL_MANIFEST),
        "captions": str(CAPTION_JSONL),
        "caption_field": "caption",
        "caption_sampling": "random",
        "random_caption_fields": ["brief", "descriptive", "analytical", "positional"],
        "target_key": "hr",
        "condition_key": "ms_hr" if is_ms else None,
        "train_degradation_sampling": "random",
        "degradation_seed": 42,
        "degradation_severity": "mild",
    })

    config["model"]["input_channels"] = 6 if is_ms else 3
    config["model"]["output_channels"] = 3
    config["model"]["decoder_upsample_mode"] = "resize_conv"
    config["model"]["use_text_conditioning"] = True

    # Use the real frozen text encoder for caption-conditioned runs.
    config["text_encoder"] = {
        "kind": "siglip",
        "model_name": "google/siglip-base-patch16-224",
        "max_tokens": 64,
    }
    config["prompts"] = {
        "null_probability": 0.4,
        "paraphrase_probability": 0.2,
        "mismatch_probability": 0.1,
    }

    config["training"].update({
        "batch_size": BATCH_BY_CAPTION_VARIANT[caption_variant]["batch"],
        "gradient_accumulation": BATCH_BY_CAPTION_VARIANT[caption_variant]["accumulation"],
        "num_workers": 8,
        "amp": True,
        "gradient_checkpointing": True,
        "auto_resume": True,
        "progress_mode": "compact",
        "progress_updates_per_epoch": 4,
        "validate_every": 1,
        "validation_limit": VALIDATION_LIMIT,
        "keep_best_and_latest": True,
        "checkpoint_metric": "val_l1",
        "checkpoint_mode": "min",
        "early_stopping_patience": 4,
        "early_stopping_min_epochs": 4,
        "early_stopping_min_delta": 0.00005,
    })
    config.setdefault("debug", {})
    config["debug"].update({"enabled": False, "print_tensor_stats": False})
    return config

CONFIGS_CAPTION = {name: build_caption_config(name) for name in CAPTION_VARIANTS_TO_RUN}
train_count = sum(record["split"] == "train" for record in records)

for variant, config in CONFIGS_CAPTION.items():
    directory = CONFIG_ROOT / variant
    directory.mkdir(parents=True, exist_ok=True)
    (directory / "template.yaml").write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    report = build_parameter_report(config, patches=train_count, world_size=1)
    print(
        variant,
        "base=", CAPTION_TO_BASE_VARIANT[variant],
        "input_channels=", config["model"]["input_channels"],
        "condition_key=", config["data"]["condition_key"],
        "core=", f"{report['core_model']['scalar_parameters']:,}",
        "joint=", f"{report['training_stages']['joint']['total_optimized_parameters']:,}",
    )

## 10. Train Caption-Conditioned Variants

Caption runs are stored under new variant names, so existing no-caption checkpoints remain intact.
If interrupted, rerun this cell; `auto_resume=True` continues from the latest checkpoint.

In [ ]:
import copy, yaml
from pathlib import Path


def select_checkpoint_for_variant(variant, stage):
    directory = RUN_ROOT / variant / stage
    candidates = [
        directory / f"{stage}_best.pt",
        directory / f"{stage}_last.pt",
        directory / f"{stage}_latest.pt",
    ]
    candidates.extend(sorted(directory.glob(f"{stage}_epoch_*.pt"), reverse=True))
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"No checkpoint for {variant}/{stage} in {directory}")


def train_caption_variant(caption_variant):
    base_variant = CAPTION_TO_BASE_VARIANT[caption_variant]
    checkpoints = {}

    if REUSE_NO_CAPTION_BASE_VAE:
        checkpoints["base"] = select_checkpoint_for_variant(base_variant, "base")
        checkpoints["vae"] = select_checkpoint_for_variant(base_variant, "vae")
        previous = checkpoints["vae"]
        print(f"{caption_variant}: reusing base checkpoint: {checkpoints['base']}")
        print(f"{caption_variant}: reusing VAE checkpoint:  {checkpoints['vae']}")
    else:
        previous = None

    for stage in STAGES_TO_RUN_CAPTION:
        config = copy.deepcopy(CONFIGS_CAPTION[caption_variant])
        output = RUN_ROOT / caption_variant / stage
        config["training"].update({
            "stage": stage,
            "epochs": EPOCHS_BY_CAPTION_VARIANT[caption_variant][stage],
            "output_dir": str(output),
            "init_checkpoint": str(previous) if previous else None,
            "resume": None,
            "auto_resume": AUTO_RESUME_TRAINING,
        })
        if stage == "joint":
            config["training"]["learning_rate"] = 1e-5
            config["training"]["loss_weights"]["adversarial"] = 0.003

        config_path = CONFIG_ROOT / caption_variant / f"{stage}.yaml"
        config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
        print(f"\n===== {caption_variant}: {stage} =====")
        run([PYTHON, "-m", "geodiff_gan.cli.train", "--config", config_path], cwd=REPOSITORY_DIR)
        previous = select_checkpoint_for_variant(caption_variant, stage)
        checkpoints[stage] = previous
        print("Selected:", previous)

    return checkpoints

CHECKPOINTS_CAPTION_BY_VARIANT = {}
for variant in CAPTION_VARIANTS_TO_RUN:
    CHECKPOINTS_CAPTION_BY_VARIANT[variant] = train_caption_variant(variant)

CHECKPOINTS_CAPTION_BY_VARIANT

## 11. Evaluate All Eight Variants with LPIPS

No-caption variants are loaded from the already-trained checkpoints. Caption variants are loaded
from the new caption-conditioned checkpoints. All are evaluated on the same test split.

In [ ]:
import json, subprocess
from pathlib import Path
import pandas as pd

# LPIPS is optional in the codebase; install it in this venv if missing.
run([PYTHON, "-m", "pip", "install", "-q", "lpips"], cwd=REPOSITORY_DIR)

COMPARISON_SPLIT = "test"
COMPARISON_LIMIT = 40      # increase to 300/1000/None for final reporting
COMPARISON_SAMPLES = 2
COMPARISON_STEPS = 20
COMPARISON_BACK_PROJECTION_STEPS = 3
COMPARISON_EVAL_ROOT = WORK_ROOT / "evaluation_caption_vs_nocaption"


def checkpoint_for_comparison(variant, stage="joint"):
    return select_checkpoint_for_variant(variant, stage)


def config_for_comparison(variant):
    config_path = CONFIG_ROOT / variant / "joint.yaml"
    if config_path.exists():
        return config_path
    template = CONFIG_ROOT / variant / "template.yaml"
    if template.exists():
        return template
    raise FileNotFoundError(f"Missing comparison config for {variant}: {config_path}")


def is_caption_variant(variant):
    return variant in CAPTION_VARIANTS_TO_RUN

EIGHT_VARIANT_RESULTS = {}
for variant in ALL_VARIANTS_FOR_COMPARISON:
    config_path = config_for_comparison(variant)
    checkpoint = checkpoint_for_comparison(variant, "joint")
    output = COMPARISON_EVAL_ROOT / variant / COMPARISON_SPLIT
    command = [
        PYTHON, "-m", "geodiff_gan.cli.evaluate",
        "--config", config_path,
        "--checkpoint", checkpoint,
        "--output", output,
        "--split", COMPARISON_SPLIT,
        "--samples", COMPARISON_SAMPLES,
        "--steps", COMPARISON_STEPS,
        "--back-projection-steps", COMPARISON_BACK_PROJECTION_STEPS,
        "--mode", "sr",
        "--device", "cuda",
        "--progress", "compact",
        "--optional-metrics",
    ]
    if COMPARISON_LIMIT is not None:
        command.extend(["--limit", COMPARISON_LIMIT])
    if not is_caption_variant(variant):
        command.append("--no-text")

    print(f"\n===== evaluate {variant} =====")
    run(command, cwd=REPOSITORY_DIR)
    metrics = json.loads((output / "metrics.json").read_text(encoding="utf-8"))
    EIGHT_VARIANT_RESULTS[variant] = metrics

rows = []
for variant in ALL_VARIANTS_FOR_COMPARISON:
    metrics = EIGHT_VARIANT_RESULTS[variant]
    rows.append({
        "variant": variant,
        "captioned": is_caption_variant(variant),
        "family": CAPTION_TO_BASE_VARIANT.get(variant, variant),
        "condition": "multispectral" if (variant.endswith("_ms") or "_ms_" in variant) else "rgb",
        "count": metrics.get("count"),
        "psnr": metrics.get("psnr"),
        "ssim": metrics.get("ssim"),
        "edge_f1": metrics.get("edge_f1"),
        "lpips": metrics.get("lpips"),
        "l1": metrics.get("l1"),
        "redegradation_l1": metrics.get("redegradation_l1"),
        "observed_lr_noise_to_signal": metrics.get("observed_lr_noise_to_signal"),
    })

comparison_table = pd.DataFrame(rows)
display(comparison_table.round(6))
COMPARISON_EVAL_ROOT.mkdir(parents=True, exist_ok=True)
comparison_csv = COMPARISON_EVAL_ROOT / "eight_variant_caption_vs_nocaption_metrics.csv"
comparison_table.to_csv(comparison_csv, index=False)
print("Saved:", comparison_csv)

## 12. Plot PSNR, SSIM, Edge F1, and LPIPS

Higher is better for PSNR, SSIM, and Edge F1. Lower is better for LPIPS.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_table = comparison_table.copy()
plot_order = ALL_VARIANTS_FOR_COMPARISON
plot_table["variant"] = pd.Categorical(plot_table["variant"], categories=plot_order, ordered=True)
plot_table = plot_table.sort_values("variant")

metric_specs = [
    ("psnr", "PSNR ?"),
    ("ssim", "SSIM ?"),
    ("edge_f1", "Edge F1 ?"),
    ("lpips", "LPIPS ?"),
]

colors = ["#4C78A8" if not captioned else "#F58518" for captioned in plot_table["captioned"]]
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()

for ax, (metric, title) in zip(axes, metric_specs):
    values = plot_table[metric].astype(float)
    bars = ax.bar(plot_table["variant"].astype(str), values, color=colors)
    ax.set_title(title, fontsize=14, weight="bold")
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylabel(metric)
    for bar, value in zip(bars, values):
        if pd.isna(value):
            label = "NA"
            y = 0
        elif metric == "psnr":
            label = f"{value:.2f}"
            y = value
        else:
            label = f"{value:.4f}"
            y = value
        ax.text(bar.get_x() + bar.get_width() / 2, y, label, ha="center", va="bottom", fontsize=8)

from matplotlib.patches import Patch
legend = [
    Patch(facecolor="#4C78A8", label="no caption"),
    Patch(facecolor="#F58518", label="caption-conditioned"),
]
fig.legend(handles=legend, loc="upper center", ncol=2, frameon=False)
fig.suptitle(
    f"GeoDiff-GAN 8-Variant Comparison on {COMPARISON_SPLIT} "
    f"(limit={COMPARISON_LIMIT}, samples={COMPARISON_SAMPLES}, steps={COMPARISON_STEPS})",
    fontsize=16,
    weight="bold",
    y=1.02,
)
plt.tight_layout()
plot_path = COMPARISON_EVAL_ROOT / "eight_variant_caption_vs_nocaption_metrics.png"
plt.savefig(plot_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved plot:", plot_path)

## 13. Paired Caption vs No-Caption Delta Table

This table shows whether captions improved or degraded each matching architecture.
For PSNR/SSIM/Edge F1 positive is better. For LPIPS negative is better.

In [ ]:
pairs = []
for caption_variant, base_variant in CAPTION_TO_BASE_VARIANT.items():
    base_row = comparison_table[comparison_table["variant"] == base_variant].iloc[0]
    cap_row = comparison_table[comparison_table["variant"] == caption_variant].iloc[0]
    pairs.append({
        "base_variant": base_variant,
        "caption_variant": caption_variant,
        "delta_psnr": cap_row["psnr"] - base_row["psnr"],
        "delta_ssim": cap_row["ssim"] - base_row["ssim"],
        "delta_edge_f1": cap_row["edge_f1"] - base_row["edge_f1"],
        "delta_lpips": cap_row["lpips"] - base_row["lpips"],
        "caption_psnr": cap_row["psnr"],
        "nocaption_psnr": base_row["psnr"],
        "caption_lpips": cap_row["lpips"],
        "nocaption_lpips": base_row["lpips"],
    })

delta_table = pd.DataFrame(pairs)
display(delta_table.round(6))
delta_csv = COMPARISON_EVAL_ROOT / "caption_delta_against_nocaption.csv"
delta_table.to_csv(delta_csv, index=False)
print("Saved:", delta_csv)

## 14. Optional Side-by-Side Output Check

This visualizes outputs that were saved during evaluation for the same test patch.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.nn import functional as F
from geodiff_gan.data import SentinelPatchDataset

SIDE_BY_SIDE_INDEX = 0
side_dataset = SentinelPatchDataset(
    FINAL_MANIFEST,
    split=COMPARISON_SPLIT,
    scale=4,
    caption_file=str(CAPTION_JSONL),
    caption_field="caption",
    caption_sampling="fixed",
    augment=False,
    random_degradation=False,
    degradation_seed=42,
    degradation_severity="mild",
)

sample = side_dataset[SIDE_BY_SIDE_INDEX]
stem = Path(sample["patch"]).stem
hr = sample["hr"]
lr = sample["lr_rgb"] if "lr_rgb" in sample else sample["lr"]
bicubic = F.interpolate(lr[None], size=hr.shape[-2:], mode="bicubic", align_corners=False)[0].clamp(0, 1)

outputs = {}
for variant in ALL_VARIANTS_FOR_COMPARISON:
    result = COMPARISON_EVAL_ROOT / variant / COMPARISON_SPLIT / f"{stem}_uncertainty.npz"
    if result.exists():
        with np.load(result) as data:
            outputs[variant] = torch.from_numpy(data["mean"]).float()
    else:
        print("missing output:", result)

panels = [(F.interpolate(lr[None], size=hr.shape[-2:], mode="nearest")[0], "Input LR"), (bicubic, "Bicubic")]
panels += [(outputs[v], v) for v in ALL_VARIANTS_FOR_COMPARISON if v in outputs]
panels += [(hr, "Target HR")]

fig, axes = plt.subplots(2, len(panels), figsize=(3.4 * len(panels), 7), squeeze=False)
for column, (image, title) in enumerate(panels):
    axes[0, column].imshow(image.clamp(0, 1).permute(1, 2, 0))
    axes[0, column].set_title(title, fontsize=9)
    axes[0, column].axis("off")
    error = (image - hr).abs().mean(0)
    axes[1, column].imshow(error, cmap="turbo", vmin=0, vmax=max(0.05, float(error.quantile(0.99))))
    axes[1, column].set_title(f"L1={float(error.mean()):.4f}", fontsize=9)
    axes[1, column].axis("off")
fig.suptitle(stem)
plt.tight_layout()
plt.show()